# 斗地主 VRPO + CentralCritic · Colab L4

使用 [AtaraxosAI/doudizhu](https://github.com/AtaraxosAI/doudizhu) 的原始 `VRPOTrainer`，固定提交 `acdcf199cf422e2b683cb22142b8e7cb33b14460`。模型 v1.3，FP32/TF32，默认 Inductor，Muon + AdamW，policy/critic 各 4 epochs，replay scale=16。

在 Colab 中选择 **运行时 → 更改运行时类型 → L4 GPU**，然后从上到下运行。自动释放实例前会等待 Drive 写入同步；同步失败时保留运行时供你处理。首次运行包括 C++ 扩展安装与模型编译；训练时长从训练循环开始计算，包含编译，在完整一轮结束后停止并保存，可能超时一轮。默认训练 120 分钟；默认每 30 分钟保存一次，并保留当前运行目录最新 10 次检查点，均可在下一格修改。

续训恢复原仓库保存的策略、CentralCritic、优化器与 iteration/步数；原仓库不保存 replay、环境状态或随机数状态，因此续训不是逐位连续回放。默认学习率等调度仍使用仓库的 iteration 规则。检查点只应来自你信任的来源。


## L4 实测结果

测试环境：Python 3.13、PyTorch 2.11.0+cu130、CUDA 13.0。后续 Colab 软件变化可能影响吞吐与显存。

两种种子复测合并吞吐 **31.42 条已交付训练轨迹/秒**；采样显存峰值 **10.16 GiB**；按 `max(NVML峰值, PyTorch保留峰值+512MiB)` 计算的最低显存余量 **12.08 GiB**。

这是本次有限候选范围内的已验证最优配置，优化目标为端到端训练吞吐。批次/minibatch/replay 容量会影响学习行为，没有验证长期收敛或棋力。调参阶段没有保存训练检查点。性能排除热身、保存和评测；Drive 保存时会暂停训练。


| 独立复测配置 (R/C/M/线程) | 轨迹/秒 | 最低保守余量 GiB |
|---|---:|---:|
| 512/256/4/1 | 31.35 | 12.08 |
| 512/256/4/4 | 31.42 | 12.08 |

R=采样并行数，C=每轮收集轨迹数，M=minibatches。线程列是请求值；原仓库将请求1自动扩展到全部CPU，本次实际为12线程，请求4则生效4线程。两种种子的测量结果按总轨迹/总时间合并。

In [ ]:
# @title 1. 用户配置
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

DRIVE_FOLDER = "/content/drive/MyDrive/doudizhu" # @param {type:"string"}
# 续训时改为上次运行打印的 RUN_NAME，并设置 RESUME_FROM="latest"。
RUN_NAME = datetime.now(ZoneInfo("Asia/Hong_Kong")).strftime("vrpo-l4-%Y%m%d-%H%M%S")
TRAINING_MINUTES = 120 # @param {type:"number"}
SAVE_EVERY_MINUTES = 30 # @param {type:"number"}
KEEP_CHECKPOINTS = 10 # @param {type:"integer"}
RESUME_FROM = "" # @param {type:"string"}
# 空字符串：新训练；"latest"：当前运行目录最新检查点；也可填完整 Drive 文件路径。
DISCONNECT_WHEN_DONE = True # @param {type:"boolean"}
SEED = 0 # @param {type:"integer"}
BEST = {'rollout': 512, 'collect': 256, 'mini_batches': 4, 'threads': 4}
# threads 是请求值；仓库会把1自动扩展为所有可用CPU（本次L4为12）。
COMMIT = "acdcf199cf422e2b683cb22142b8e7cb33b14460"
REPOSITORY = "https://github.com/AtaraxosAI/doudizhu.git"
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in (".", "..")
assert Path(DRIVE_FOLDER).is_absolute()
print("本次运行目录:", Path(DRIVE_FOLDER) / RUN_NAME)


In [ ]:
# @title 2. 挂载 Google Drive、读取目录及检查点
from google.colab import drive
import re

drive.mount("/content/drive")
DRIVE_ROOT = Path(DRIVE_FOLDER)
assert DRIVE_ROOT.resolve().is_relative_to(Path("/content/drive")), "请选择已挂载的 Google Drive 内的目录"
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = DRIVE_ROOT / RUN_NAME / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("Drive 文件夹内容（最多20项）:")
for entry in sorted(DRIVE_ROOT.iterdir())[:20]:
    print(entry.name)
existing = sorted((p for p in CHECKPOINT_DIR.glob("ckpt_*.pt") if p.is_file() and re.fullmatch(r"ckpt_\d+\.pt", p.name)),
                  key=lambda p: int(p.stem.split("_")[1]))
print("当前运行检查点:", [p.name for p in existing])
print("检查点目录:", CHECKPOINT_DIR)


In [ ]:
# @title 3. 安装固定版本训练代码（保留 Colab CUDA/PyTorch）
import os, subprocess, sys, json, math

assert math.isfinite(TRAINING_MINUTES) and TRAINING_MINUTES > 0
assert math.isfinite(SAVE_EVERY_MINUTES) and SAVE_EVERY_MINUTES > 0
assert isinstance(KEEP_CHECKPOINTS, int) and not isinstance(KEEP_CHECKPOINTS, bool) and KEEP_CHECKPOINTS >= 1
os.environ.update(WANDB_DISABLED="true", PYTORCH_ALLOC_CONF="expandable_segments:True",
    TORCHINDUCTOR_CACHE_DIR="/content/vrpo-inductor-cache", TORCHINDUCTOR_COMPILE_THREADS="2",
    OMP_NUM_THREADS=str(BEST["threads"]), MKL_NUM_THREADS=str(BEST["threads"]), NANOPOKERLIB_BUILD_JOBS="4")
gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], text=True)
print(gpu)
assert "L4" in gpu, "请先选择 L4 GPU 运行时"
REPO = Path("/content/doudizhu-vrpo-" + COMMIT[:12])
if not REPO.exists():
    subprocess.run(["git", "clone", REPOSITORY, str(REPO)], check=True)
    subprocess.run(["git", "checkout", "--detach", COMMIT], cwd=REPO, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == COMMIT
assert not subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO, text=True).strip(), "训练代码已修改，请使用干净目录"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cmake", "ninja", "pybind11", "tyro", "numpy>=2", "wheel", "setuptools"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "--no-deps", "-e", str(REPO / "nanopokerlib")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO)], check=True)
print(subprocess.check_output([sys.executable, "-c", "import torch; print('torch:', torch.__version__, 'CUDA:', torch.version.cuda)"], text=True))


下一格只添加时间控制和 Drive 保存规则，训练本身继续调用原仓库算法。先在本地临时文件写完，再复制到 Drive 的 `.partial` 文件，完整复制后才公布文件名并清理旧检查点；失败时保留旧检查点。仅清理当前 `RUN_NAME/checkpoints` 中符合 `ckpt_数字.pt` 的文件，其他文件不动。


In [ ]:
%%writefile /content/vrpo_timed_training.py
"""Colab wall-time/Drive adapter; VRPO and checkpoint contents remain upstream."""
import json
import math
import os
from pathlib import Path
import re
import shutil
import signal
import sys
import tempfile
import time


def checkpoints(directory):
    return sorted((p for p in Path(directory).glob('ckpt_*.pt')
                   if p.is_file() and re.fullmatch(r'ckpt_\d+\.pt', p.name)),
                  key=lambda p: int(p.stem.split('_')[1]))


def publish_checkpoint(manager, state, directory, keep):
    if not isinstance(keep, int) or isinstance(keep, bool) or keep < 1:
        raise ValueError('keep_checkpoints must be a positive integer')
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / f'ckpt_{state.iteration}.pt'
    if target.exists():
        raise FileExistsError(f'Refusing to overwrite {target}')
    pending = directory / (target.name + '.partial')
    try:
        with tempfile.TemporaryDirectory(prefix='vrpo-checkpoint-') as tmp:
            local = Path(tmp) / target.name
            manager.save_from_state(state, checkpoint_path=local, upload_to_wandb=False)
            shutil.copyfile(local, pending)
            if local.stat().st_size != pending.stat().st_size:
                raise IOError('Incomplete checkpoint copy to Drive')
            os.replace(pending, target)
    finally:
        pending.unlink(missing_ok=True)
    # Only prune this run's checkpoints after the new file is fully published.
    for old in checkpoints(directory)[:-keep]:
        old.unlink()
    return target


def timed_workflow(settings):
    import torch
    from marvel.training.workflow import TrainingWorkflow
    for key in ('training_minutes', 'save_every_minutes'):
        if not math.isfinite(settings[key]) or settings[key] <= 0:
            raise ValueError(f'{key} must be positive and finite')
    keep = settings['keep_checkpoints']
    if not isinstance(keep, int) or isinstance(keep, bool) or keep < 1:
        raise ValueError('keep_checkpoints must be a positive integer')

    class TimedWorkflow(TrainingWorkflow):
        def __init__(self, cfg):
            super().__init__(cfg, game=cfg.game, checkpoint_dir=settings['checkpoint_dir'])
            print(f'CPU threads: requested={settings["threads"]}, effective={torch.get_num_threads()}', flush=True)
            self.logger.progress_bar.total = None
            self.stop_requested = False
            self.last_saved_iteration = 0
            self.last_saved_time = time.monotonic()
            self.previous_handlers = {}

        def load_checkpoint_payload(self, checkpoint_path, *, device):
            # Stage on CPU so loading does not double the live GPU model/optimizer memory.
            return super().load_checkpoint_payload(checkpoint_path, device=torch.device('cpu'))

        def checkpoint_due(self):
            return time.monotonic() - self.last_saved_time >= settings['save_every_minutes'] * 60

        def save_checkpoint(self, state):
            if state.iteration == self.last_saved_iteration:
                return
            path = publish_checkpoint(self.checkpoint_manager, state, self.checkpoint_dir, keep)
            self.last_saved_iteration = state.iteration
            self.last_saved_time = time.monotonic()
            print(f'DRIVE_CHECKPOINT {path}', flush=True)

        def iterations(self):
            start = time.monotonic()
            self.last_saved_time = start
            def request_stop(signum, frame):
                self.stop_requested = True
                print('Stop requested; finish current iteration and save.', flush=True)
            for sig in (signal.SIGINT, signal.SIGTERM):
                self.previous_handlers[sig] = signal.signal(sig, request_stop)
            try:
                for iteration in super().iterations():
                    yield iteration
                    if self.stop_requested or time.monotonic() - start >= settings['training_minutes'] * 60:
                        self.save_checkpoint(self._state)
                        print(f'TRAINING_COMPLETE iteration={iteration} elapsed_seconds={time.monotonic()-start:.1f}', flush=True)
                        break
                else:
                    if self._state is not None and self.current_iteration is not None:
                        self.save_checkpoint(self._state)
            finally:
                for sig, handler in self.previous_handlers.items():
                    signal.signal(sig, handler)
    return TimedWorkflow


def validate_resume_payload(payload):
    required = {'agent_resume_state_dict', 'agent_state_dict', 'critic_state_dict',
                'agent_optimizer_state_dict', 'critic_optimizer_state_dict',
                'iteration', 'train_config'}
    missing = required - payload.keys()
    if missing or payload.get('train_config', {}).get('algo_name') != 'vrpo':
        raise ValueError(f'Expected a full VRPO training checkpoint; missing keys: {sorted(missing)}')


def main(settings):
    import torch
    from marvel.config import TrainConfig
    from marvel.training.vrpo import VRPOTrainer
    from marvel.utils import configure_torch
    torch.set_num_threads(settings['threads'])
    torch.set_num_interop_threads(1)
    directory = Path(settings['checkpoint_dir'])
    previous = checkpoints(directory)
    resume = settings.get('resume_from', '')
    if resume == 'latest':
        if not previous:
            raise FileNotFoundError(f'No checkpoints in {directory}')
        resume = str(previous[-1])
    elif previous and not resume:
        raise ValueError('Run folder already contains checkpoints: set resume_from="latest" or choose a new run folder')
    if resume and not Path(resume).is_file():
        raise FileNotFoundError(resume)
    if resume:
        payload = torch.load(resume, map_location='cpu', weights_only=True)
        validate_resume_payload(payload)
        if previous and int(payload['iteration']) < int(previous[-1].stem.split('_')[1]):
            raise ValueError('Resuming an older checkpoint requires a new run folder to preserve newer checkpoints')
        del payload
    cfg = TrainConfig(algo_name='vrpo', device='cuda', total_iterations=2_000_000_000,
        checkpoint_interval=-1, evaluation_interval=-1, param_stats_interval=-1,
        rollout_batch_size=settings['rollout'], collect_batch_size=settings['collect'],
        mini_batches=settings['mini_batches'], replay_buffer_scale=16,
        resume_from=resume or None, resume_optimizer=True, seed=settings.get('seed', 0),
        logging_dir=settings['logging_dir'])
    configure_torch(cfg)
    VRPOTrainer(cfg, runtime_factory=timed_workflow(settings)).run()


if __name__ == '__main__':
    main(json.loads(Path(sys.argv[1]).read_text()))


In [ ]:
# @title 5. 启动训练，到时保存并结束
import signal

settings = dict(BEST, training_minutes=TRAINING_MINUTES, save_every_minutes=SAVE_EVERY_MINUTES,
    keep_checkpoints=KEEP_CHECKPOINTS, resume_from=RESUME_FROM, checkpoint_dir=str(CHECKPOINT_DIR),
    logging_dir=str(Path("/content/vrpo-training-logs") / RUN_NAME), seed=SEED)
settings_path = Path("/content/vrpo-training-settings.json")
settings_path.write_text(json.dumps(settings, indent=2))
(DRIVE_ROOT / RUN_NAME / "training-settings.json").write_text(json.dumps(dict(settings, commit=COMMIT), indent=2))
env = dict(os.environ, PYTHONPATH=str(REPO))
try:
    process = subprocess.Popen([sys.executable, "/content/vrpo_timed_training.py", str(settings_path)], cwd=REPO, env=env)
    try:
        returncode = process.wait()
    except KeyboardInterrupt:
        print("请求在当前完整训练轮结束后保存，请等待；强制删除运行时无法触发保存。")
        process.send_signal(signal.SIGINT)
        returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(f"训练失败，退出码 {returncode}；请检查日志。旧检查点保留。将按配置同步Drive并释放资源。")
    print("训练已结束，检查点:")
    for path in sorted(CHECKPOINT_DIR.glob("ckpt_*.pt"), key=lambda p: int(p.stem.split("_")[1])):
        print(path, f"{path.stat().st_size / 2**20:.1f} MiB")
    print("继续训练时使用 RUN_NAME =", repr(RUN_NAME), "以及 RESUME_FROM = 'latest'")
finally:
    if DISCONNECT_WHEN_DONE:
        from google.colab import drive, runtime
        drive.flush_and_unmount(timeout_ms=120000)
        runtime.unassign()


In [ ]:
# @title 6. 手动释放 L4（失败或提前结束后也应执行）
from google.colab import drive, runtime
drive.flush_and_unmount(timeout_ms=120000)
runtime.unassign()


显存余量来自短时实际训练测量，无法保证未来所有局面、软件版本或长时间运行都不 OOM。发生 OOM 时先重启运行时，减小 collect/rollout 或增加 minibatches，并重新验证；不要通过反复立即重试消耗额度。

Google Drive 首次挂载需要你在 Colab 浏览器授权。此 notebook 已检查 Python 语法与保存/计时逻辑；挂载与实际 Drive 写入必须在你的授权会话中执行。强制关闭实例、断电或无法完成当前轮时，只能恢复到最近一次已发布检查点。
